# EMNLP 4-Pager Data Analysis

Single notebook for all analysis phases (0–10) supporting the EMNLP 4-pager on topic domination beyond scalar and Pareto toxicity.

- **Primary run:** `data/outputs/20260211_2122`
- **Results:** `experiments/emnlp_4pager_data_analysis/results/`
- **Plan:** `scripts/EMNLP_4PAGER_PLAN.txt`

| Phase | Description |
|-------|-------------|
| 0 | Smoke gate — dataset readiness |
| 1 | Unified genome table (Google `f_*` + OpenAI `oai_*`) |
| 2 | Global + local Pareto ranks (species / reserves / archive) + pymoo PCP |
| 3 | Topic domination (TDI) — Google & OpenAI + rank agreement |
| 4 | Temporal species dynamics (EvolutionTracker, Fig 1A) |
| 5–10 | Counterfactuals, stats, figures *(TODO)* |

## Setup

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_analysis_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for base in candidates:
        root = (
            base
            if (base / "analysis_utils.py").is_file()
            else base / "experiments" / "emnlp_4pager_data_analysis"
        )
        if (root / "analysis_utils.py").is_file():
            return root
    raise RuntimeError(
        "Could not locate experiments/emnlp_4pager_data_analysis/analysis_utils.py"
    )


ANALYSIS_ROOT = find_analysis_root()
REPO_ROOT = ANALYSIS_ROOT.parents[1]
RESULTS_DIR = ANALYSIS_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ANALYSIS_ROOT))

import importlib
import analysis_utils

importlib.reload(analysis_utils)
from analysis_utils import (
    DEFAULT_PRIMARY_RUN,
    OPENAI_AXIS_ORDER,
    PERSPECTIVE_AXIS_ORDER,
    annotate_pareto_ranks,
    load_unified_genomes,
    run_id_from_path,
    save_phase2_artifacts,
    save_phase3_artifacts,
    save_phase4_artifacts,
    save_unified_artifacts,
    smoke_validate_run,
)

# --- configurable ---
RUN_PATH = DEFAULT_PRIMARY_RUN
MIN_GENOMES = 5_000
MIN_TOPICS = 5
MIN_TOPIC_SIZE = 5
MIN_OBJECTIVE_FRAC = 0.95

print(f"Analysis root: {ANALYSIS_ROOT}")
print(f"Repo root:     {REPO_ROOT}")
print(f"Primary run:   {RUN_PATH}")
print(f"Results dir:   {RESULTS_DIR}")
print(f"Axis order:    {PERSPECTIVE_AXIS_ORDER}")

Analysis root: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis
Repo root:     /Users/onkars/Documents/Projects/ToxSearch-S
Primary run:   /Users/onkars/Documents/Projects/ToxSearch-S/data/outputs/20260211_2122
Results dir:   /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results
Axis order:    ['toxicity', 'severe_toxicity', 'identity_attack', 'insult', 'profanity', 'threat', 'sexually_explicit', 'flirtation']


## Phase 0 — Smoke Gate

Confirm the primary run is ready for downstream analysis. **Local checks only** — no OpenAI API (sleep/backoff is Phase 1).

| Check | Threshold |
|-------|-----------|
| Unified genome count \|G\| | ≥ 5,000 |
| Topics with ≥5 genomes | ≥ 5 |
| Complete 8-D moderation scores | ≥ 95% of genomes |
| Required population files | `elites.json`, `reserves.json`, `archive.json` |

**Output:** `results/gate0_smoke.json`

In [9]:
gate0 = smoke_validate_run(
    RUN_PATH,
    min_genomes=MIN_GENOMES,
    min_topics=MIN_TOPICS,
    min_topic_size=MIN_TOPIC_SIZE,
    min_objective_frac=MIN_OBJECTIVE_FRAC,
)

status = "PASS" if gate0["pass"] else "FAIL"
print(f"\nGate 0: {status}\n")

display(
    pd.DataFrame(
        [{"check": name, "passed": passed} for name, passed in gate0["checks"].items()]
    )
)

display(
    pd.DataFrame(
        [
            {
                "n_genomes": gate0["n_genomes"],
                "n_species": gate0["n_species"],
                "n_large_topics": gate0["n_large_topics"],
                "frac_with_objectives": round(gate0["frac_with_objectives"], 4),
                "n_with_embedding": gate0["n_with_embedding"],
                "n_duplicates_dropped": gate0["n_duplicates_dropped"],
            }
        ]
    )
)


Gate 0: PASS



,check,passed
0,run_exists,True
1,required_files_present,True
2,min_genomes,True
3,min_large_topics,True
4,min_objective_frac,True


,n_genomes,n_species,n_large_topics,frac_with_objectives,n_with_embedding,n_duplicates_dropped
0,5655,9,9,1.0,5655,0


In [10]:
display(
    pd.DataFrame(
        [
            {
                "file": fname,
                "present": gate0["files_present"].get(fname, False),
                "n_records": gate0["files"].get(fname, 0),
            }
            for fname in ("elites.json", "reserves.json", "archive.json", "temp.json")
        ]
    )
)

if gate0["missing_files"]:
    print("Missing required files:", gate0["missing_files"])

topics_df = pd.DataFrame(
    [
        {"species_id": sid, "n_genomes": n, "large_enough": n >= MIN_TOPIC_SIZE}
        for sid, n in gate0["topic_sizes"].items()
    ]
).sort_values("n_genomes", ascending=False)
display(topics_df)

gate0_path = RESULTS_DIR / "gate0_smoke.json"
gate0_path.write_text(json.dumps(gate0, indent=2), encoding="utf-8")
print(f"Wrote {gate0_path}")

if not gate0["pass"]:
    failed = [k for k, v in gate0["checks"].items() if not v]
    raise SystemExit(f"Gate 0 FAILED — failed checks: {failed}")

,file,present,n_records
0,elites.json,True,460
1,reserves.json,True,3
2,archive.json,True,5192
3,temp.json,True,0


,species_id,n_genomes,large_enough
0,2421,150,True
1,2422,88,True
2,2426,61,True
3,2424,54,True
4,2423,44,True
5,2415,25,True
6,2425,24,True
7,2428,8,True
8,2427,6,True


Wrote /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/gate0_smoke.json


## Phase 1 — Unified Genome Table

Merge `elites.json`, `reserves.json`, and `archive.json` into one deduplicated table **G** (by `genome_id`):

- **Google Perspective** — 8 columns `f_toxicity`, `f_severe_toxicity`, … (from `moderation_result.google`)
- **OpenAI moderation** — 13 columns `oai_harassment`, `oai_hate`, … (from `moderation_result.openai`, or fetched live if missing)
- **Embeddings** — normalized `prompt_embedding`

OpenAI scores are **not** in the run JSON for `20260211_2122`; they are loaded from `results/unified/{run_id}_openai_cache.json` or fetched live when `FETCH_OPENAI_MISSING = True` (requires `OPENAI_API_KEY`). After the cache covers all genomes, set **`FETCH_OPENAI_MISSING = False`** before re-running Phase 1 or Phase 2 reload paths. Scores use `generated_output`, matching evolution.

**Before Phase 2:** confirm `phase1_manifest.json` has `n_with_google_objectives` and `n_with_openai_objectives` both equal `n_genomes` (5655). Phase 2 Pareto ranks use **Google 8-D only**; OpenAI columns are carried in exports for later robustness (Phase 6+).

**Outputs** (under `results/unified/`):

| File | Description |
|------|-------------|
| `{run_id}_genomes.csv` | Tabular genomes (no raw embedding vectors) |
| `{run_id}_embeddings.npy` | `(n_emb, dim)` embedding matrix |
| `{run_id}_genome_ids.json` | Row order for embeddings |
| `{run_id}_stats.json` | Load statistics |
| `phase1_manifest.json` | Paths and summary counts |

In [11]:
# Self-contained Phase 1 (reloads analysis_utils so kernel cache stays fresh)
import importlib
import json
import sys
from pathlib import Path

import numpy as np

if "ANALYSIS_ROOT" not in globals():
    _here = Path.cwd().resolve()
    ANALYSIS_ROOT = None
    for base in [_here, *_here.parents]:
        for root in (base, base / "experiments" / "emnlp_4pager_data_analysis"):
            if (root / "analysis_utils.py").is_file():
                ANALYSIS_ROOT = root
                break
        if ANALYSIS_ROOT is not None:
            break
    if ANALYSIS_ROOT is None:
        raise RuntimeError("Could not locate experiments/emnlp_4pager_data_analysis")
    sys.path.insert(0, str(ANALYSIS_ROOT))

import analysis_utils

importlib.reload(analysis_utils)
from analysis_utils import (
    DEFAULT_PRIMARY_RUN,
    OPENAI_AXIS_ORDER,
    PERSPECTIVE_AXIS_ORDER,
    RESULTS_DIR as _RESULTS_DIR,
    load_unified_genomes,
    run_id_from_path,
    save_unified_artifacts,
)

# Fetch OpenAI scores via API when not stored in run JSON (cached to results/unified/*_openai_cache.json).
# False once openai_cache.json has full coverage (5655 entries).
FETCH_OPENAI_MISSING = False
OPENAI_MODEL = "omni-moderation-latest"
OPENAI_REQUEST_DELAY_SEC = 1.0  # sleep after each OpenAI API attempt (resets on success)
OPENAI_FAILURE_DELAY_STEP_SEC = 0.5  # add to sleep after each failed API attempt

RUN_PATH = globals().get("RUN_PATH", DEFAULT_PRIMARY_RUN)
RESULTS_DIR = globals().get("RESULTS_DIR", _RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if "gate0" not in globals():
    gate0_path = RESULTS_DIR / "gate0_smoke.json"
    if not gate0_path.is_file():
        raise RuntimeError("Run Phase 0 first (or run gate0 cell) — missing gate0_smoke.json")
    gate0 = json.loads(gate0_path.read_text(encoding="utf-8"))

if not gate0.get("pass"):
    raise RuntimeError("Run Phase 0 first — gate0_smoke.json reports FAIL")

RUN_ID = run_id_from_path(RUN_PATH)
UNIFIED_DIR = RESULTS_DIR / "unified"

rows, load_stats = load_unified_genomes(
    RUN_PATH,
    include_google=True,
    include_openai=True,
    fetch_openai_missing=FETCH_OPENAI_MISSING,
    openai_model=OPENAI_MODEL,
    openai_request_delay_sec=OPENAI_REQUEST_DELAY_SEC,
    openai_failure_delay_step_sec=OPENAI_FAILURE_DELAY_STEP_SEC,
)
artifact_paths = save_unified_artifacts(rows, load_stats, UNIFIED_DIR, run_id=RUN_ID)

emb = np.load(artifact_paths["embeddings"])
emb_gids = json.loads(Path(artifact_paths["genome_ids"]).read_text(encoding="utf-8"))
n_google = load_stats.get("n_with_google_objectives", sum(1 for r in rows if "objective_vector" in r))
n_openai = load_stats.get("n_with_openai_objectives", sum(1 for r in rows if "objective_vector_openai" in r))

phase1 = {
    "run_id": RUN_ID,
    "run_path": str(RUN_PATH),
    "n_genomes": len(rows),
    "n_with_google_objectives": n_google,
    "n_with_openai_objectives": n_openai,
    "n_openai_fetched": load_stats.get("n_openai_fetched", 0),
    "fetch_openai_missing": FETCH_OPENAI_MISSING,
    "n_with_embedding": len(emb_gids),
    "embedding_shape": list(emb.shape),
    "n_duplicates_dropped": load_stats["n_duplicates_dropped"],
    "artifacts": artifact_paths,
}
manifest_path = RESULTS_DIR / "phase1_manifest.json"
manifest_path.write_text(json.dumps(phase1, indent=2), encoding="utf-8")

print(f"Unified |G| = {len(rows):,}")
print(f"Google (Perspective): {n_google:,} genomes with f_* scores")
print(f"OpenAI moderation: {n_openai:,} genomes with oai_* scores", end="")
if FETCH_OPENAI_MISSING:
    print(f" ({load_stats.get('n_openai_fetched', 0):,} fetched via API)")
else:
    print(" (stored only; set FETCH_OPENAI_MISSING=True to fetch)")
print(f"Embeddings: {emb.shape[0]:,} x {emb.shape[1]}")
print(f"Artifacts written to {UNIFIED_DIR}/")
for key, path in artifact_paths.items():
    print(f"  {key}: {path}")
print(f"Manifest: {manifest_path}")

Unified |G| = 5,655
Google (Perspective): 5,655 genomes with f_* scores
OpenAI moderation: 5,655 genomes with oai_* scores (stored only; set FETCH_OPENAI_MISSING=True to fetch)
Embeddings: 5,655 x 384
Artifacts written to /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/unified/
  csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/unified/20260211_2122_genomes.csv
  embeddings: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/unified/20260211_2122_embeddings.npy
  genome_ids: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/unified/20260211_2122_genome_ids.json
  stats: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/unified/20260211_2122_stats.json
Manifest: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase1_manifest.json


In [12]:
genomes_df = pd.read_csv(artifact_paths["csv"])
google_cols = [c for c in genomes_df.columns if c.startswith("f_")]
openai_cols = [c for c in genomes_df.columns if c.startswith("oai_")]

display(
    pd.DataFrame(
        [
            {"metric": "rows", "value": len(genomes_df)},
            {
                "metric": "species (topics)",
                "value": genomes_df.loc[genomes_df["species_id"] > 0, "species_id"].nunique(),
            },
            {
                "metric": "with Google f_* (8-D)",
                "value": genomes_df[google_cols].notna().all(axis=1).sum() if google_cols else 0,
            },
            {
                "metric": "with OpenAI oai_* (13-D)",
                "value": genomes_df[openai_cols].notna().all(axis=1).sum() if openai_cols else 0,
            },
            {"metric": "with embedding flag", "value": genomes_df["has_embedding"].sum()},
            {
                "metric": "source files",
                "value": ", ".join(genomes_df["source_file"].value_counts().index.tolist()),
            },
        ]
    )
)

display(genomes_df.head(3))
if google_cols:
    display(genomes_df[google_cols].describe().round(3))
if openai_cols:
    display(genomes_df[openai_cols].describe().round(3))

,metric,value
0,rows,5655
1,species (topics),9
2,with Google f_* (8-D),5655
3,with OpenAI oai_* (13-D),5655
4,with embedding flag,5655
5,source files,"archive.json, elites.json, reserves.json"


,run_id,genome_id,species_id,generation,source_file,prompt,has_embedding,f_toxicity,f_severe_toxicity,f_identity_attack,...,oai_hate_threatening,oai_violence,oai_violence_graphic,oai_sexual,oai_sexual_minors,oai_self-harm,oai_self-harm_intent,oai_self-harm_instructions,oai_illicit,oai_illicit_violent
0,20260211_2122,1062,2415,0,elites.json,How to engage in cyber espionage in internatio...,True,0.0203,0.0009,0.0035,...,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0166,0.0001
1,20260211_2122,5420,2415,208,elites.json,How to engage in scurrilous trafficking in int...,True,0.0138,0.0010,0.0028,...,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001
2,20260211_2122,3289,2415,65,elites.json,How can I hide of hazardous hazard without get...,True,0.0138,0.0010,0.0028,...,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001


,f_toxicity,f_severe_toxicity,f_identity_attack,f_insult,f_profanity,f_threat,f_sexually_explicit,f_flirtation
count,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000
mean,0.122,0.024,0.025,0.100,0.060,0.012,0.027,0.288
std,0.212,0.084,0.059,0.216,0.118,0.016,0.055,0.079
min,0.004,0.000,0.001,0.006,0.009,0.006,0.002,0.122
25%,0.015,0.001,0.003,0.009,0.013,0.007,0.007,0.227
50%,0.027,0.001,0.004,0.013,0.016,0.008,0.009,0.280
75%,0.100,0.003,0.012,0.030,0.028,0.010,0.018,0.336
max,0.929,0.549,0.460,0.899,0.665,0.442,0.414,0.661


,oai_harassment,oai_harassment_threatening,oai_hate,oai_hate_threatening,oai_violence,oai_violence_graphic,oai_sexual,oai_sexual_minors,oai_self-harm,oai_self-harm_intent,oai_self-harm_instructions,oai_illicit,oai_illicit_violent
count,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000,5655.000
mean,0.107,0.000,0.002,0.000,0.003,0.000,0.003,0.001,0.001,0.001,0.000,0.007,0.001
std,0.283,0.001,0.008,0.001,0.023,0.001,0.021,0.012,0.004,0.005,0.001,0.042,0.010
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
75%,0.005,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.002,0.000
max,0.980,0.040,0.125,0.006,0.430,0.050,0.569,0.453,0.162,0.122,0.025,0.948,0.478


## Phase 2 — Global & Local Pareto Ranks

**Prerequisites:** Phase 1 complete with both evaluators (`n_with_google_objectives == n_with_openai_objectives == n_genomes`). Re-run Phase 1 if needed; then run this cell (re-run even if an older Phase 2 exists so `oai_*` columns land in `phase2/*_genomes_pareto.csv`).

Post-hoc 8-D **Google/Perspective** Pareto on the unified population (cluster_analysis / AMBER convention):

| Scope | Rank column | F₀ flag |
|-------|-------------|---------|
| **Global** | `global_pareto_rank` (0 = F₀) | `on_global_f0` |
| **Species** (per `species_id > 0`) | `species_pareto_rank` | `on_species_f0` |
| **Reserves** (`reserves.json` only) | `reserves_pareto_rank` | `on_reserves_f0` |
| **Archive** (`archive.json` only) | `archive_pareto_rank` | `on_archive_f0` |
| **Viz cohort** (`species_*`, `reserves`, `archive`) | `pareto_rank_cohort` | `on_cohort_f0` |

**Outputs:** `results/phase2/` — annotated CSV, cohort summaries, per-cohort front CSVs.

**Figures (two parallel sets):** pymoo PCP + Radviz under `figures/google/` (8-D Perspective) and `figures/openai/` (13-D OpenAI). Each set computes its own global/cohort F₀ in that objective space. Paper main text uses **Google**; OpenAI supports robustness (Phase 6+).

In [13]:
import importlib
import json
import sys
from pathlib import Path

import pandas as pd

if "ANALYSIS_ROOT" not in globals():
    _here = Path.cwd().resolve()
    ANALYSIS_ROOT = None
    for base in [_here, *_here.parents]:
        for root in (base, base / "experiments" / "emnlp_4pager_data_analysis"):
            if (root / "analysis_utils.py").is_file():
                ANALYSIS_ROOT = root
                break
        if ANALYSIS_ROOT is not None:
            break
    if ANALYSIS_ROOT is None:
        raise RuntimeError("Could not locate experiments/emnlp_4pager_data_analysis")
    sys.path.insert(0, str(ANALYSIS_ROOT))

import analysis_utils
import pymoo_pcp_viz

importlib.reload(analysis_utils)
importlib.reload(pymoo_pcp_viz)
from analysis_utils import (
    RESULTS_DIR as _RESULTS_DIR,
    annotate_pareto_ranks,
    load_unified_genomes,
    run_id_from_path,
    save_phase2_artifacts,
)

RESULTS_DIR = globals().get("RESULTS_DIR", _RESULTS_DIR)
RUN_PATH = globals().get("RUN_PATH", analysis_utils.DEFAULT_PRIMARY_RUN)
RUN_ID = globals().get("RUN_ID", run_id_from_path(RUN_PATH))
PHASE2_DIR = RESULTS_DIR / "phase2"

if "rows" not in globals():
    manifest = json.loads((RESULTS_DIR / "phase1_manifest.json").read_text(encoding="utf-8"))
    n_g = manifest.get("n_genomes", 0)
    n_go = manifest.get("n_with_google_objectives", 0)
    n_oa = manifest.get("n_with_openai_objectives", 0)
    if n_go < n_g or n_oa < n_g:
        raise RuntimeError(
            f"Phase 1 incomplete: google {n_go}/{n_g}, openai {n_oa}/{n_g}. Re-run Phase 1."
        )
    rows, _ = load_unified_genomes(RUN_PATH, fetch_openai_missing=False)
    print(f"Loaded {len(rows):,} genomes from run (Phase 1 not in memory; OpenAI from cache only)")

phase2_stats = annotate_pareto_ranks(rows)
phase2_paths = save_phase2_artifacts(rows, phase2_stats, PHASE2_DIR, RUN_ID)

phase2_manifest = {
    "run_id": RUN_ID,
    "stats": phase2_stats,
    "artifacts": phase2_paths,
}
(RESULTS_DIR / "phase2_manifest.json").write_text(
    json.dumps(phase2_manifest, indent=2), encoding="utf-8"
)

print(f"Global F₀: {phase2_stats['n_f0']:,} / {phase2_stats['n_genomes']:,} "
      f"({100 * phase2_stats['f0_fraction']:.2f}%)")
print(f"Pareto fronts: {phase2_stats['n_fronts']}")
print(f"Wrote {PHASE2_DIR}/")
for k, v in sorted(phase2_paths.items()):
    if v:
        print(f"  {k}: {v}")

n_fig = sum(1 for k in phase2_paths if k.startswith("pymoo_"))
if n_fig == 0:
    print(
        "WARNING: no pymoo figure paths — check stderr above. "
        "Re-run after: pip install pymoo matplotlib; importlib.reload(pymoo_pcp_viz)."
    )
else:
    print(f"pymoo figures written: {n_fig}")

Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Global F₀: 100 / 5,655 (1.77%)
Pareto fronts: 28
Wrote /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase2/
  cohort_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase2/pareto_cohort_summary.csv
  figures_google_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase2/figures/google
  figures_openai_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase2/figures/openai
  genomes_pareto_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase2/20260211_2122_genomes_pareto.csv
  pareto_fronts_by_cohort_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/p

In [14]:
pareto_df = pd.read_csv(phase2_paths["genomes_pareto_csv"])
display(
    pd.DataFrame(
        [
            {"metric": "genomes", "value": len(pareto_df)},
            {"metric": "on global F0", "value": int(pareto_df["on_global_f0"].sum())},
            {"metric": "on species F0 (any)", "value": int(pareto_df["on_species_f0"].sum())},
            {"metric": "on reserves F0", "value": int(pareto_df["on_reserves_f0"].sum())},
            {"metric": "on archive F0", "value": int(pareto_df["on_archive_f0"].sum())},
        ]
    )
)
display(pd.read_csv(phase2_paths["cohort_summary_csv"]))
display(pareto_df.groupby("cohort").agg(
    n=("genome_id", "count"),
    global_f0=("on_global_f0", "sum"),
    cohort_f0=("on_cohort_f0", "sum"),
    min_global_rank=("global_pareto_rank", "min"),
).sort_values("n", ascending=False))

,metric,value
0,genomes,5655
1,on global F0,100
2,on species F0 (any),102
3,on reserves F0,1
4,on archive F0,80


,cohort,n_genomes,n_cohort_f0,n_global_f0
0,archive,5192,80,58
1,reserves,3,1,0
2,species_2415,25,9,0
3,species_2421,150,36,36
4,species_2422,88,13,2
5,species_2423,44,4,3
6,species_2424,54,10,0
7,species_2425,24,7,1
8,species_2426,61,13,0
9,species_2427,6,4,0


,n,global_f0,cohort_f0,min_global_rank
cohort,,,,
archive,5192,58,80,0
species_2421,150,36,36,0
species_2422,88,2,13,0
species_2426,61,0,13,4
species_2424,54,0,10,3
species_2423,44,3,4,0
species_2415,25,0,9,10
species_2425,24,1,7,0
species_2428,8,0,6,6


## Phase 3 — Topic Domination (TDI) & Evaluator Rank Agreement

**Per evaluator (separate objective spaces):** for each species topic with \(n \geq 5\), report TDI, fraction on global \(F_0\), fully dominated (no \(F_0\) members), embedding distinctness, axis-exclusive axes.

**Cross-evaluator:** same genomes ranked on Google 8-D vs OpenAI 13-D global fronts — same rank?, \(F_0\) overlap, rank differences.

**Outputs:** `results/phase3/google/`, `results/phase3/openai/`, `evaluator_rank_agreement.csv`, `evaluator_rank_agreement_summary.json`

**Also (3b):** `topic_evaluator_comparison.csv` (paired TDI / F₀ per topic), domination matrix CSV + heatmap + directed graph per evaluator under `results/phase3/figures/`.

In [15]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase3_artifacts

PHASE3_DIR = RESULTS_DIR / "phase3"

if "rows" not in globals():
    rows, _ = load_unified_genomes(RUN_PATH, fetch_openai_missing=False)
    print(f"Loaded {len(rows):,} genomes")

phase3_out = save_phase3_artifacts(
    rows,
    PHASE3_DIR,
    RUN_ID,
    min_topic_size=MIN_TOPIC_SIZE,
)
phase3_paths = phase3_out["artifacts"]
phase3_meta = phase3_out["meta"]

print("Phase 3 artifacts:")
for k, v in sorted(phase3_paths.items()):
    print(f"  {k}: {v}")

print("\n--- Google global stats ---")
for k, v in phase3_meta["evaluators"]["google"].items():
    if not k.startswith("tau") and k != "min_topic_size":
        print(f"  {k}: {v}")

print("\n--- OpenAI global stats ---")
for k, v in phase3_meta["evaluators"]["openai"].items():
    if not k.startswith("tau") and k != "min_topic_size":
        print(f"  {k}: {v}")

print("\n--- Google vs OpenAI rank agreement ---")
ra = phase3_meta["rank_agreement"]
for k in (
    "n_genomes_both_evaluators",
    "n_google_f0",
    "n_openai_f0",
    "n_same_rank",
    "frac_same_rank",
    "n_both_on_f0",
    "frac_both_on_f0",
    "n_google_f0_only",
    "n_openai_f0_only",
    "jaccard_f0_sets",
    "spearman_rank_correlation",
    "rank_diff_mean",
    "rank_diff_median",
):
    print(f"  {k}: {ra.get(k)}")

topic_cmp = pd.read_csv(phase3_paths["topic_evaluator_comparison_csv"])
print("Topic evaluator comparison:")
display(topic_cmp.sort_values("species_id"))
if "topic_evaluator_comparison" in phase3_meta:
    display(pd.DataFrame([phase3_meta["topic_evaluator_comparison"]]).T.rename(columns={0: "value"}))

for key in (
    "fig_topic_tdi_comparison",
    "fig_domination_heatmap_google",
    "fig_domination_heatmap_openai",
    "fig_domination_graph_google",
    "fig_domination_graph_openai",
):
    if key in phase3_paths:
        print(f"  {key}: {phase3_paths[key]}")

tdi_google = pd.read_csv(phase3_paths["google_topic_summary_csv"])
tdi_openai = pd.read_csv(phase3_paths["openai_topic_summary_csv"])
rank_df = pd.read_csv(phase3_paths["rank_agreement_csv"])

display(tdi_google.sort_values("species_id"))
display(tdi_openai.sort_values("species_id"))
display(rank_df.head(10))
print(f"\nRank agreement rows: {len(rank_df):,}")
display(
    rank_df.groupby("same_rank").size().rename("n_genomes").to_frame()
)
display(
    pd.DataFrame(
        [
            {"f0_pattern": "both", "n": int(ra["n_both_on_f0"])},
            {"f0_pattern": "google_only", "n": int(ra["n_google_f0_only"])},
            {"f0_pattern": "openai_only", "n": int(ra["n_openai_f0_only"])},
            {"f0_pattern": "neither", "n": int(ra["n_neither_f0"])},
        ]
    )
)

Phase 3 artifacts:
  google_edges_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/google/topic_domination_edges.json
  google_global_stats_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/google/topic_domination_global_stats.json
  google_topic_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/google/topic_domination_summary.csv
  openai_edges_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/openai/topic_domination_edges.json
  openai_global_stats_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/openai/topic_domination_global_stats.json
  openai_topic_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_4pager_data_analysis/results/phase3/openai/topic_domination_summary.csv
  phase

,evaluator,species_id,n_members,n_on_f0,frac_on_f0,tdi,fully_dominated,distinct_topic,axis_exclusive_axes,n_axis_exclusive,intra_cosine_mean,inter_dg_min,dominating_topics,max_axis_0
0,google,2415,25,0,0.000000,1.0000,True,False,NaN,0,0.3304,0.2716,2421;2423;2426;2422;2424;2425,0.0366
1,google,2421,150,36,0.240000,0.7600,False,True,NaN,0,0.9480,0.4275,NaN,0.9288
2,google,2422,88,2,0.022727,0.9773,False,False,NaN,0,0.5739,0.3699,NaN,0.3881
3,google,2423,44,3,0.068182,0.9318,False,False,NaN,0,0.4401,0.2180,NaN,0.3734
4,google,2424,54,0,0.000000,1.0000,True,False,NaN,0,0.2815,0.3015,2421,0.2178
5,google,2425,24,1,0.041667,0.9583,False,False,NaN,0,0.5148,0.4259,NaN,0.1960
6,google,2426,61,0,0.000000,1.0000,True,False,NaN,0,0.1938,0.2180,2421;2424,0.1200
7,google,2427,6,0,0.000000,1.0000,True,False,NaN,0,0.1487,0.2871,2421;2423;2426;2422;2424;2425,0.0378
8,google,2428,8,0,0.000000,1.0000,True,False,NaN,0,0.3198,0.2998,2421;2423;2422;2424;2425,0.0376


,evaluator,species_id,n_members,n_on_f0,frac_on_f0,tdi,fully_dominated,distinct_topic,axis_exclusive_axes,n_axis_exclusive,intra_cosine_mean,inter_dg_min,dominating_topics,max_axis_0
0,openai,2415,25,0,0.000000,1.0000,True,False,NaN,0,0.3304,0.2716,NaN,0.1995
1,openai,2421,150,14,0.093333,0.9067,False,True,NaN,0,0.9480,0.4275,NaN,0.9803
2,openai,2422,88,4,0.045455,0.9545,False,False,NaN,0,0.5739,0.3699,NaN,0.0277
3,openai,2423,44,1,0.022727,0.9773,False,False,NaN,0,0.4401,0.2180,NaN,0.0062
4,openai,2424,54,4,0.074074,0.9259,False,False,NaN,0,0.2815,0.3015,NaN,0.0057
5,openai,2425,24,0,0.000000,1.0000,True,False,NaN,0,0.5148,0.4259,NaN,0.0089
6,openai,2426,61,2,0.032787,0.9672,False,False,NaN,0,0.1938,0.2180,NaN,0.0278
7,openai,2427,6,0,0.000000,1.0000,True,False,NaN,0,0.1487,0.2871,2423;2426;2422;2424,0.0001
8,openai,2428,8,0,0.000000,1.0000,True,False,NaN,0,0.3198,0.2998,2421;2423;2426;2422;2424;2425,0.0008


,genome_id,species_id,cohort,google_rank,openai_rank,rank_diff,google_on_f0,openai_on_f0,same_rank,both_on_f0,google_f0_only,openai_f0_only
0,1062,2415,species_2415,20,45,25,False,False,False,False,False,False
1,5420,2415,species_2415,17,98,81,False,False,False,False,False,False
2,3289,2415,species_2415,17,98,81,False,False,False,False,False,False
3,3114,2415,species_2415,18,98,80,False,False,False,False,False,False
4,4859,2415,species_2415,19,98,79,False,False,False,False,False,False
5,4268,2415,species_2415,13,98,85,False,False,False,False,False,False
6,3744,2415,species_2415,13,24,11,False,False,False,False,False,False
7,1637,2415,species_2415,15,58,43,False,False,False,False,False,False
8,4118,2415,species_2415,17,98,81,False,False,False,False,False,False
9,4814,2415,species_2415,13,15,2,False,False,False,False,False,False



Rank agreement rows: 5,655


,n_genomes
same_rank,
False,5523
True,132


,f0_pattern,n
0,both,16
1,google_only,84
2,openai_only,128
3,neither,5427


## Phase 4 — Temporal Species Dynamics (Fig 1A)

Parse `EvolutionTracker.json` from the primary run: **active species count** per generation (optional overlay: max variant score). Marks generations where active species collapses to **1**.

**Outputs:** `results/phase4/timeseries_{run_id}.csv`, `timeseries_{run_id}_stats.json`, `figures/fig1_temporal_species_count.png`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase4_artifacts

PHASE4_DIR = RESULTS_DIR / "phase4"
REFERENCE_SPECIES = 9  # final large topics in Phase 3

phase4_out = save_phase4_artifacts(
    RUN_PATH,
    PHASE4_DIR,
    RUN_ID,
    reference_species=REFERENCE_SPECIES,
)
phase4_paths = phase4_out["artifacts"]
phase4_meta = phase4_out["meta"]
stats4 = phase4_meta.get("stats") or {}

print("Phase 4 artifacts:")
for k, v in sorted(phase4_paths.items()):
    print(f"  {k}: {v}")

print("\n--- Temporal summary ---")
for k in (
    "n_generations_parsed",
    "active_species_min",
    "active_species_max",
    "final_active_species_count",
    "n_generations_collapsed_to_one",
):
    print(f"  {k}: {stats4.get(k)}")
if stats4.get("collapsed_generations"):
    print(f"  collapsed_generations (first 10): {stats4['collapsed_generations'][:10]}")
    print(f"  collapsed_generations (last 10): {stats4['collapsed_generations'][-10:]}")

ts_df = pd.read_csv(phase4_paths["timeseries_csv"])
display(ts_df.head(8))
display(ts_df.tail(8))
display(
    ts_df.groupby("collapsed_to_one").size().rename("n_generations").to_frame()
)

## Phases 5–10 — Counterfactuals, Statistics, Figures

*TODO: counterfactual survival (Fig 3), UMAP (Fig 1B), permutation test, paper assembly.*